In [139]:
import pandas as pd
import sqlite3

<h2>Создайте подключение к базе данных с помощью sqlite3 библиотеки.</h2>

In [140]:
conn = sqlite3.connect('../data/checking-logs.sqlite')

<h2>Получите схему таблицы test</h2>

In [141]:
shema = pd.read_sql('PRAGMA table_info(test);', conn)
print(shema)

   cid             name       type  notnull dflt_value  pk
0    0              uid       TEXT        0       None   0
1    1          labname       TEXT        0       None   0
2    2  first_commit_ts  TIMESTAMP        0       None   0
3    3    first_view_ts  TIMESTAMP        0       None   0


<h2>Возьмите только первые десять строк таблицы, test чтобы увидеть, как она выглядит.</h2>

In [142]:
preview = pd.read_sql('SELECT * FROM test LIMIT 10;', conn)
print(preview)

       uid   labname             first_commit_ts               first_view_ts
0   user_1    laba04  2020-04-26 17:06:18.462708  2020-04-26 21:53:59.624136
1   user_1   laba04s  2020-04-26 17:12:11.843671  2020-04-26 21:53:59.624136
2   user_1    laba05  2020-05-02 19:15:18.540185  2020-04-26 21:53:59.624136
3   user_1    laba06  2020-05-17 16:26:35.268534  2020-04-26 21:53:59.624136
4   user_1   laba06s  2020-05-20 12:23:37.289724  2020-04-26 21:53:59.624136
5   user_1  project1  2020-05-14 20:56:08.898880  2020-04-26 21:53:59.624136
6  user_10    laba04  2020-04-25 08:24:52.696624  2020-04-18 12:19:50.182714
7  user_10   laba04s  2020-04-25 08:37:54.604222  2020-04-18 12:19:50.182714
8  user_10    laba05  2020-05-01 19:27:26.063245  2020-04-18 12:19:50.182714
9  user_10    laba06  2020-05-19 11:39:28.885637  2020-04-18 12:19:50.182714


<h2>Найти минимальное значение дельты между первым коммитом и дедлайном соответствующей лабораторной для всех пользователей.</h2>

- Использовать только один SQL-запрос.
- Для этого нужно соединить таблицу test с таблицей deadlines.
- Разницу следует отображать в часах.
- Не учитывать лабораторную project1, так как у неё длинный дедлайн — это выброс.
- Значение должно быть сохранено в фрейме данных df_min с соответствующим uid.

In [143]:
query = """ 
        SELECT t.uid, t.labname,
            MIN((CAST (strftime('%s', first_commit_ts) AS INT) - deadlines) / 3600) AS min_delta_hours
        FROM test t JOIN deadlines d ON t.labname == d.labs
        WHERE t.labname != 'project1'
        """

In [144]:
df_min = pd.read_sql(query, conn)

In [145]:
df_min.head()

,uid,labname,min_delta_hours
0,user_30,laba04,-202


<h2>Сделайте то же самое для максимального значения, но используйте только один запрос. Имя фрейма данных — df_max.</h2>

In [146]:
query = """ 
        SELECT t.uid, t.labname,
            MAX((CAST (strftime('%s', first_commit_ts) AS INT) - deadlines) / 3600) AS max_delta_hours
        FROM test t JOIN deadlines d ON t.labname == d.labs
        WHERE t.labname != 'project1'
        """

In [147]:
df_max = pd.read_sql(query, conn)
df_max.head()

,uid,labname,max_delta_hours
0,user_25,laba04s,-2


<h2>Сделайте то же самое, но для среднего значения. Используйте только один запрос. На этот раз ваш фрейм данных не должен содержать столбец uid. Имя фрейма данных — df_avg.</h2>

In [148]:
query = """ 
        SELECT 
            AVG((CAST (strftime('%s', first_commit_ts) AS INT) - deadlines) / 3600) AS avg_delta_hours
        FROM test t JOIN deadlines d ON t.labname == d.labs
        WHERE t.labname != 'project1'
"""

In [149]:
df_avg = pd.read_sql(query, conn)
df_avg.head()

,avg_delta_hours
0,-89.125


<h2>Мы хотим проверить гипотезу о том, что пользователи, посетившие ленту новостей всего несколько раз, имеют меньшую разницу между первым коммитом и дедлайном. Для этого рассчитаем коэффициент корреляции между количеством просмотров страниц и разницей.</h2>

- Используя только один запрос, создайте таблицу со следующими столбцами: «uid», «avg_diff» и «pageviews».
- «uid» — это UID, которые существуют в test.
- «avg_diff» — это средняя разница между первым коммитом и крайним сроком лабораторных работ для каждого пользователя.
- «Просмотры страниц» — это количество посещений новостной ленты на одного пользователя.
- Лабораторию project1в расчет не брать.
- Сохраните его в фрейме данных views_diff.
- Используйте метод Pandas corr()для расчета коэффициента корреляции между количеством просмотров страниц и разницей.

In [150]:
query = """
SELECT t.uid,
        AVG( 
            (CAST (strftime('%s', first_commit_ts) AS INT) - deadlines) / 3600
            ) as avg_diff,
        (SELECT COUNT(*) FROM pageviews WHERE t.uid = pageviews.uid) as page_views
FROM test t INNER JOIN deadlines d ON t. labname = d. labs
WHERE t. labname != 'project1'
GROUP BY uid
"""

In [151]:
views_diff = pd.read_sql(query, conn)
views_diff.head()

,uid,avg_diff,page_views
0,user_1,-64.400000,28
1,user_10,-74.800000,89
2,user_14,-159.000000,143
3,user_17,-61.600000,47
4,user_18,-5.666667,3


In [152]:
views_diff.corr(numeric_only=True)

,avg_diff,page_views
avg_diff,1.000000,-0.279736
page_views,-0.279736,1.000000


<h2>Закройте соединение.</h2>

In [153]:
conn.close()